# **SCAPES: simplified tutorial**

In this notebook you will be able to train a SCAPES model with mostly default settings, generate sound examples and export your model. 

For additional details and configurations on the process see `tutorial.ipynb`.

## **1. Setup**

### **1.1. Installation**

If on Colab, this cell installs the SCAPES repository and its dependencies. If ran locally, please install the dependencies manually following our Readme and run this notebook from the quickstart directory.

In [ ]:
import os
import subprocess
import sys

def SCAPES_setup():
    IN_COLAB = "google.colab" in sys.modules
    if IN_COLAB == True:
        answer = "in Colab."
    else:
        answer = "locally."
    print("You are running SCAPES",answer)

    # Fixed branch for now
    BRANCH = "main"  # branch cloned on Colab
    if IN_COLAB:
        # from google.colab import drive

        # clone once: the folder /content/SCAPES is kept if the cell runs again
        if not os.path.isdir("/content/SCAPES"):
            subprocess.run(["git", "clone", "--recurse-submodules", "--branch", BRANCH,
                            "https://github.com/cordutie/SCAPES.git", "/content/SCAPES"], check=True)
        os.chdir("/content/SCAPES")
        # drive.mount("/content/drive")

        # the install steps of the README (install:env), without the virtual environment
        python_m = [sys.executable, "-m"]
        subprocess.run(python_m + ["pip", "install", "--upgrade", "pip"], check=True)
        subprocess.run(python_m + ["pip", "install", "-r", "requirements.txt",
                                "--extra-index-url", "https://download.pytorch.org/whl/cu128"], check=True)
        subprocess.run(python_m + ["pip", "install", "--no-deps", "msclap==1.3.4",
                                "magenta-rt @ git+https://github.com/magenta/magenta-realtime@694a545e4ba0b88bf1150137b129582166d3e07f"],
                    check=True)
        subprocess.run(python_m + ["pip", "install", "music2latent==0.1.8"], check=True)
        subprocess.run(python_m + ["pip", "install", "-e", "./external/audio_codec_wrappers"], check=True)
        subprocess.run([sys.executable, "external/audio_codec_wrappers/scripts/fetch_weights.py", "--codec", "spectrostream"],
                    check=True)
        REPO = os.getcwd()
    else:
        # walk up from the working directory to the repository root
        REPO = os.getcwd()
        while not os.path.exists(os.path.join(REPO, "SCAPES", "pipeline.py")):
            parent = os.path.dirname(REPO)
            if parent == REPO:
                raise FileNotFoundError("Open this notebook from inside the SCAPES repository.")
            REPO = parent

    # the repository must be importable as SCAPES and quickstart
    if REPO not in sys.path:
        sys.path.insert(0, REPO)

    print("SCAPES is ready!")

In [ ]:
SCAPES_setup()

### **1.2. Parameters**

Point to the paths of the audio corpus to be used for training, the config file, and where the model will be built.

If you have already ran the data preparation with the same configurations before, use the same model folder so the prepared data is reused.

In [ ]:
from wrappers import parameter_wrapper

corpus_path = "/home/esteban/Documents/projects/dev_SCAPES/old_stuff/datasets/datitos_olivier"
build_path  = "/home/esteban/Documents/projects/dev_SCAPES/models/oli_music2latent_10hz_1000ms"
config_path = "/home/esteban/Documents/projects/dev_SCAPES/config_music2latent.gin"

CORPUS_PATH, BUILD_PATH, CONFIG_PATH, TRAINING, MODEL_PATH = parameter_wrapper(corpus_path, build_path, config_path)

### **1.3. Real-Time Factor check (optional)**

Test your model configurations in your machine to know if it can run in real-time. Note that the test is run using medium quality, so there is a bit of margin in case you either want to run it on worse (around twice as fast) or better quality (around twice as slow).

In [ ]:
from wrappers import test_cpu, test_gpu

test_gpu(CONFIG_PATH, TRAINING)
# test_cpu(CONFIG_PATH, TRAINING)

## **2. Dataprep**

Encodes every WAV file of the wav folder with the codec and computes the semantic annotations (CLAP) that the model is conditioned on. The result goes to `BUILD_PATH/dataset/`. If it is already there for the same config and wavs, it is reused and nothing runs.

In [ ]:
from quickstart.wrappers import data_preparation

data_preparation(CORPUS_PATH, BUILD_PATH, CONFIG_PATH)

In [ ]:
# Free the memory of the step before, without a session reset
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

## **4. Training**

Trains the model and saves it in `MODEL_PATH` (`models/<size>_<ms>ms` in the build folder). Training runs until the validation loss stops improving, at most `training.max_epochs` (400). It also writes the semantics folder from the cherry-picking CSV, which the generation step needs. If the dataset is too short for a validation split, the last checkpoint is used. A model folder that already exists is refused: change the size or the memory in the config, or use another build folder.

In [ ]:
from quickstart.wrappers import training_loop

training_loop(CORPUS_PATH, BUILD_PATH, CONFIG_PATH)

In [ ]:
# Free the memory of the step before, without a session reset
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

## **5. Generation**

Uses your own audio file (`AUDIO_PATH`). The model is loaded from `MODEL_PATH` (the best checkpoint, or the last one when training had no validation split). The first 10 seconds of the file are used.

In [ ]:
from IPython.display import Audio, display

from SCAPES.data.config_loader import load_training_config
from SCAPES.pipeline import model_folder

from SCAPES.inference.FlowInference import FlowInference

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
engine = FlowInference(model_dir=MODEL_PATH, device=device, verbose=False, checkpoint="best")
print("Loaded model from", MODEL_PATH)


### 6.1. Resynthesis (teacher-forced)

Reconstructs the audio from its own latent frames: the model gets the ground-truth atoms of the file as context, so this measures how well the model reproduces what it was trained on.

In [ ]:
from SCAPES.inference.FlowInference import run_resynthesis_pipeline

audio_path = "/home/esteban/Documents/projects/dev_SCAPES/models/microtex_music2latent/dataset/raw/applause.wav"
AUDIO_PATH = os.environ.get("SCAPES_AUDIO_PATH", audio_path)  # file for the resynthesis (section 6)

output = run_resynthesis_pipeline(engine=engine, audio_path=AUDIO_PATH, duration=10, play=True,
                                  TF=True, NFE=NFE, cfg_scale=CFG_SCALE)


### 6.2. Semantic reconstruction (partial teacher-forcing)

Only the first atoms of the file are given; the rest is generated from the CLAP embedding of the file, computed the way the dataprep of this model computed it (the context length and random extension of the build folder's specs). Same semantic content, different acoustic realization.

In [ ]:
output = run_resynthesis_pipeline(engine=engine, audio_path=AUDIO_PATH, duration=10, play=True,
                                  TF="partial", NFE=NFE, cfg_scale=CFG_SCALE)
